# 03 — Collaborative Filtering with TruncatedSVD

**Purpose:** Factorise a sparse user-item ratings matrix into lower-dimensional latent representations.

This notebook uses a per-user holdout split so test ratings are not used to fit the model. Results will vary with the split seed and hyperparameters.

## 1. Load ratings and create a per-user train/test split

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
# Works when launched from the Spectra root or from the notebooks folder.
if not (PROJECT_ROOT / "datasets" / "ml-1m").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "datasets" / "ml-1m"

movies = pd.read_csv(
    DATA_DIR / "movies.dat", sep="::", engine="python",
    names=["movie_id", "title", "genres"], encoding="latin-1"
)
ratings = pd.read_csv(
    DATA_DIR / "ratings.dat", sep="::", engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"], encoding="latin-1"
)
users = pd.read_csv(
    DATA_DIR / "users.dat", sep="::", engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"], encoding="latin-1"
)
print(f"Movies: {len(movies):,} | Ratings: {len(ratings):,} | Users: {len(users):,}")

from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Keep users with at least 2 ratings so each can contribute to train and test.
counts = ratings.groupby("user_id").size()
eligible = counts[counts >= 2].index
model_ratings = ratings[ratings.user_id.isin(eligible)].copy()

train_parts, test_parts = [], []
for _, group in model_ratings.groupby("user_id", sort=False):
    train_g, test_g = train_test_split(
        group, test_size=0.2, random_state=42
    )
    train_parts.append(train_g)
    test_parts.append(test_g)

train = pd.concat(train_parts, ignore_index=True)
test = pd.concat(test_parts, ignore_index=True)

user_ids = np.sort(train.user_id.unique())
movie_ids = np.sort(train.movie_id.unique())
user_to_idx = {v:i for i,v in enumerate(user_ids)}
movie_to_idx = {v:i for i,v in enumerate(movie_ids)}

train_rows = train.user_id.map(user_to_idx).to_numpy()
train_cols = train.movie_id.map(movie_to_idx).to_numpy()
R_train = csr_matrix(
    (train.rating.to_numpy(dtype=float), (train_rows, train_cols)),
    shape=(len(user_ids), len(movie_ids))
)
print("Train matrix:", R_train.shape, "| train:", len(train), "| test:", len(test))

Movies: 3,883 | Ratings: 1,000,209 | Users: 6,040
Train matrix: (6040, 3679) | train: 797758 | test: 202451


## 2. Fit TruncatedSVD

In [2]:
N_COMPONENTS = min(50, min(R_train.shape) - 1)
svd = TruncatedSVD(n_components=N_COMPONENTS, n_iter=7, random_state=42)
user_factors = svd.fit_transform(R_train)
movie_factors = svd.components_.T

print("Latent dimensions:", N_COMPONENTS)
print("Explained variance ratio (sum):", svd.explained_variance_ratio_.sum())

Latent dimensions: 50
Explained variance ratio (sum): 0.3401408946294309


## 3. Predict held-out ratings

In [3]:
known_test = test[
    test.user_id.isin(user_to_idx) & test.movie_id.isin(movie_to_idx)
].copy()
u = known_test.user_id.map(user_to_idx).to_numpy()
i = known_test.movie_id.map(movie_to_idx).to_numpy()
pred = np.sum(user_factors[u] * movie_factors[i], axis=1)
pred = np.clip(pred, 1.0, 5.0)

mae = mean_absolute_error(known_test.rating, pred)
rmse = mean_squared_error(known_test.rating, pred) ** 0.5
print(f"Evaluated test ratings: {len(known_test):,}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

Evaluated test ratings: 202,416
MAE:  2.1921
RMSE: 2.4555


## 4. Compare against a global-mean baseline

In [4]:
global_mean = train.rating.mean()
baseline_pred = np.full(len(known_test), global_mean)
baseline_mae = mean_absolute_error(known_test.rating, baseline_pred)
baseline_rmse = mean_squared_error(known_test.rating, baseline_pred) ** 0.5

pd.DataFrame([
    {"model":"Global mean baseline", "MAE":baseline_mae, "RMSE":baseline_rmse},
    {"model":"TruncatedSVD", "MAE":mae, "RMSE":rmse}
])

,model,MAE,RMSE
0,Global mean baseline,0.935601,1.118059
1,TruncatedSVD,2.192105,2.455466


## 5. Caveats

This is a simple latent-factor baseline, not a bias-aware production recommender. It does not include user/movie bias terms or implicit-feedback modelling. Random per-user holdout measures rating prediction on known users/items; it does not measure new-user or new-movie cold-start performance.